# BHS_vs_PSC_26grid — BayesHalvingSearchCV vs PatternSearchCV (2026-07-18)

The real benchmark deliverable (BayesHalvingSearchCV_SPEC.md section 10), after code is green (204 tests passing) and the from-scratch `GPProposer` is validated against Optuna's `GPSampler` (`GP_Validation_vs_Optuna.ipynb`, Experiment 12 in `EXPERIMENTS.md`). Runs in the plain package `.venv` — no torch needed anywhere in this notebook.

Same official grid, same `TimeSeriesSplit(5)`, same MAE scoring, same pipeline cell as every other benchmark notebook in this project. Primary arms, both `random_state=0`, `subsample='stratified'`, zones `(0.005, 0.01, 0.1, 1.0)`, `n_starts=1` (directly comparable to the reference rows in the spec's section 0):

1. `BayesHalvingSearchCV(n_iter=25, n_starts=1)`
2. `PatternSearchCV` current defaults (patient, `n_starts=1`) — fresh run, same session, for a same-machine wall-clock pairing.

Reference numbers this is trying to beat (Optuna GP at 100% data, Experiment 4): 15 evaluations, 15.00 full-fit equivalents, best MAE 805.730. PatternSearchCV's own best (Experiment 10): 22 evaluations, 5.09 equivalents, best MAE 805.038.

In [1]:
# --- Data pipeline: byte-for-byte replication of the prototype notebook ---
import time
import numpy as np
import pandas as pd

t0 = time.time()
trainBench = pd.read_csv(r"C:\FILES\Code\Benchmarking\train.csv")  # notebook: /home/dell/train.csv

SplitPoint = int(len(trainBench.index) * 0.8)
print("SplitPoint: ", SplitPoint)

df = trainBench

# int64 -> int32; object -> category -> numeric codes (Date included, as in the prototype)
Int64columns = df.select_dtypes(["int64"]).columns
df[Int64columns] = df[Int64columns].astype(np.int32)
cat_columns = df.select_dtypes(["object"]).columns
df[cat_columns] = df[cat_columns].astype("category")
df[cat_columns] = df[cat_columns].apply(lambda x: x.cat.codes)

# the five weather columns the prototype drops (note the dataset's own 'kM' typo)
df = df.drop("Max_Gust_SpeedKm_h", axis=1)
df = df.drop("CloudCover", axis=1)
df = df.drop("Max_VisibilityKm", axis=1)
df = df.drop("Min_VisibilitykM", axis=1)
df = df.drop("Mean_VisibilityKm", axis=1)

# positional 80/20 chronological split
trainBench, validBench = df.iloc[:SplitPoint, :], df.iloc[SplitPoint:, :]
print("Training Set shape", trainBench.shape)
print("Validation Set shape", validBench.shape)
del df

# feature mask: everything but the target (alphabetical order, incl. Date codes
# and NumberOfCustomers, exactly as the prototype had it)
mask = trainBench.columns.difference(["NumberOfSales"])
trainDataset_X = trainBench[mask]
trainDataset_y = trainBench["NumberOfSales"]
validBench_X = validBench[mask]
validBench_y = validBench["NumberOfSales"]
print("Feature Columns:")
print(list(mask))
del trainBench, validBench
print(f"Pipeline done in {time.time() - t0:.1f} s")

SplitPoint:  418416


C:\Users\Home\AppData\Local\Temp\ipykernel_13180\4255920482.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_columns = df.select_dtypes(["object"]).columns


Training Set shape (418416, 31)
Validation Set shape (104605, 31)
Feature Columns:
['AssortmentType', 'Date', 'Events', 'HasPromotions', 'IsHoliday', 'IsOpen', 'Max_Dew_PointC', 'Max_Humidity', 'Max_Sea_Level_PressurehPa', 'Max_TemperatureC', 'Max_Wind_SpeedKm_h', 'Mean_Dew_PointC', 'Mean_Humidity', 'Mean_Sea_Level_PressurehPa', 'Mean_TemperatureC', 'Mean_Wind_SpeedKm_h', 'Min_Dew_PointC', 'Min_Humidity', 'Min_Sea_Level_PressurehPa', 'Min_TemperatureC', 'NearestCompetitor', 'NumberOfCustomers', 'Precipitationmm', 'Region', 'Region_AreaKM2', 'Region_GDP', 'Region_PopulationK', 'StoreID', 'StoreType', 'WindDirDegrees']
Pipeline done in 9.9 s


In [2]:
# --- common setup: grid, CV, zones ladder [0.5%, 1%, 10%, 100%] ---
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import TimeSeriesSplit
from pattern_search_cv import BayesHalvingSearchCV, PatternSearchCV

X, y = trainDataset_X, trainDataset_y
N_features = X.shape[1]
ZONES = [0.005, 0.01, 0.1, 1.0]

def make_clf():
    return ExtraTreesRegressor(n_estimators=240, max_features=max(1, N_features - 15),
                               max_depth=16, n_jobs=1, random_state=0)

param_grid = {
    "max_features": [2, 3, 4],
    "n_estimators": list(range(10, 261, 10)),
    "max_depth": [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17],
}
tscv = TimeSeriesSplit(n_splits=5)


def _summarize(arm, search, wall):
    res = search.cv_results_
    evals = []
    for p, sc, nr in zip(res["params"], res["mean_test_score"], res["n_resources"]):
        key = (p["max_features"], p["n_estimators"], p["max_depth"], int(nr))
        evals.append({"key": key, "mae": -sc})
    out = {
        "arm": arm, "wall": wall,
        "n_fits": len(res["params"]),
        "equiv": float(np.sum(np.asarray(res["n_resources"]) / len(y))),
        "best": search.best_params_, "best_mae": -search.best_score_,
        "zones_used": sorted(set(int(v) for v in res["n_resources"])),
        "evals": evals,
    }
    print(f"{arm:22s}: {out['n_fits']} evals, {out['equiv']:.2f} equiv, "
         f"{wall:.1f} s wall, best {out['best']} MAE {out['best_mae']:.3f}")
    return out


def run_bhs(n_iter=25, n_starts=1):
    search = BayesHalvingSearchCV(make_clf(), param_grid,
                                  scoring="neg_mean_absolute_error", cv=tscv,
                                  n_jobs=-1, data_zones=ZONES,
                                  subsample="stratified", random_state=0,
                                  verbose=0, n_iter=n_iter, n_starts=n_starts)
    t0 = time.time()
    search.fit(X, y)
    wall = time.time() - t0
    out = _summarize("BayesHalvingSearchCV", search, wall)
    out["search_history"] = search.search_history_
    out["n_cache_hits"] = search.n_cache_hits_
    out["local_optima"] = search.local_optima_
    print(f"  cache hits: {out['n_cache_hits']}, local optima: {len(out['local_optima'])}")
    return out


def run_psc(n_starts=1):
    search = PatternSearchCV(make_clf(), param_grid,
                             scoring="neg_mean_absolute_error", cv=tscv,
                             n_jobs=-1, data_zones=ZONES,
                             subsample="stratified", random_state=0,
                             verbose=0, n_starts=n_starts)
    t0 = time.time()
    search.fit(X, y)
    wall = time.time() - t0
    return _summarize("PatternSearchCV (patient, defaults)", search, wall)

In [3]:
BHS = run_bhs(n_iter=25, n_starts=1)

C:\FILES\Code\Benchmarking\Working_on_Train_Set\V2025\pattern-search-cv\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter length_scale is close to the specified lower bound 0.05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


C:\FILES\Code\Benchmarking\Working_on_Train_Set\V2025\pattern-search-cv\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter length_scale is close to the specified lower bound 0.05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


C:\FILES\Code\Benchmarking\Working_on_Train_Set\V2025\pattern-search-cv\.venv\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter length_scale is close to the specified lower bound 0.05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


BayesHalvingSearchCV  : 28 evals, 3.89 equiv, 993.2 s wall, best {'max_features': 4, 'n_estimators': 150, 'max_depth': 17} MAE 805.730
  cache hits: 0, local optima: 1


In [4]:
PSC = run_psc(n_starts=1)

PatternSearchCV (patient, defaults): 22 evals, 5.09 equiv, 1157.6 s wall, best {'max_features': 4, 'n_estimators': 130, 'max_depth': 17} MAE 805.038


In [5]:
# --- BHS vs PSC comparison, zones [0.5, 1, 10, 100]%, n_starts=1 ---
print("=" * 78)
print(f"zones ladder {ZONES} (subsample='stratified'), n_starts=1")
print(f"{'':22s}{'BayesHalvingSearchCV':>24s}{'PatternSearchCV':>20s}")
print(f"{'evaluations':22s}{BHS['n_fits']:>24d}{PSC['n_fits']:>20d}")
print(f"{'full-fit equivalents':22s}{BHS['equiv']:>24.2f}{PSC['equiv']:>20.2f}")
print(f"{'wall-clock (s)':22s}{BHS['wall']:>24.1f}{PSC['wall']:>20.1f}")
print(f"{'BHS/PSC wall ratio':22s}{'':>24s}{BHS['wall'] / PSC['wall']:>20.3f}")
bhs_point = (BHS['best']['max_features'], BHS['best']['n_estimators'], BHS['best']['max_depth'])
psc_point = (PSC['best']['max_features'], PSC['best']['n_estimators'], PSC['best']['max_depth'])
print(f"{'best point':22s}{str(bhs_point):>24s}{str(psc_point):>20s}")
print(f"{'best CV MAE':22s}{BHS['best_mae']:>24.3f}{PSC['best_mae']:>20.3f}")
print(f"{'zones used (rows)':22s}{str(BHS['zones_used']):>24s}{str(PSC['zones_used']):>20s}")
print()
print(f"reference: Optuna GP 100%-data (Exp. 4): 15 evals, 15.00 equiv, MAE 805.730")
print(f"reference: PatternSearchCV best-so-far (Exp. 10): 22 evals, 5.09 equiv, MAE 805.038")

zones ladder [0.005, 0.01, 0.1, 1.0] (subsample='stratified'), n_starts=1
                          BayesHalvingSearchCV     PatternSearchCV
evaluations                                 28                  22
full-fit equivalents                      3.89                5.09
wall-clock (s)                           993.2              1157.6
BHS/PSC wall ratio                                           0.858
best point                        (4, 150, 17)        (4, 130, 17)
best CV MAE                            805.730             805.038
zones used (rows)        [2093, 41842, 418416]      [2093, 418416]

reference: Optuna GP 100%-data (Exp. 4): 15 evals, 15.00 equiv, MAE 805.730
reference: PatternSearchCV best-so-far (Exp. 10): 22 evals, 5.09 equiv, MAE 805.038


In [6]:
# --- BHS trial-by-trial history (start index, params, fraction, MAE) ---
for h in BHS["search_history"]:
    p = h["params"]
    print(f"start={h['start']:d} trial={h['trial']:>3d} "
         f"({p['max_features']},{p['n_estimators']},{p['max_depth']}) "
         f"frac={h['fraction']:.4f} MAE={-h['score']:.3f} event={h['event']}")

start=0 trial=  1 (3,130,11) frac=0.0050 MAE=1143.711 event=trial
start=0 trial=  2 (3,230,6) frac=0.0050 MAE=1402.234 event=trial
start=0 trial=  3 (3,120,11) frac=0.0050 MAE=1149.787 event=trial
start=0 trial=  4 (3,150,14) frac=0.0050 MAE=1086.105 event=trial
start=0 trial=  5 (3,200,17) frac=0.0050 MAE=1070.509 event=trial
start=0 trial=  6 (2,180,17) frac=0.0050 MAE=1203.483 event=trial
start=0 trial=  7 (4,180,17) frac=0.0050 MAE=949.853 event=trial
start=0 trial=  8 (4,240,17) frac=0.0050 MAE=959.604 event=trial
start=0 trial=  9 (4,200,15) frac=0.0050 MAE=962.547 event=trial
start=0 trial= 10 (4,50,17) frac=0.0050 MAE=968.463 event=trial
start=0 trial= 11 (4,120,17) frac=0.0050 MAE=953.545 event=trial
start=0 trial= 12 (4,110,14) frac=0.0050 MAE=972.013 event=trial
start=0 trial= 13 (4,150,16) frac=0.0050 MAE=956.159 event=trial
start=0 trial= 14 (4,10,13) frac=0.0050 MAE=1123.677 event=trial
start=0 trial= 15 (4,200,17) frac=0.0050 MAE=954.131 event=trial
start=0 trial= 16 (4,

## Summary

**BayesHalvingSearchCV beats both reference points on full-fit equivalents.**

| | evaluations | full-fit equiv | best MAE | wall-clock |
|---|---|---|---|---|
| Optuna GP, 100% data (Exp. 4) | 15 | 15.00 | 805.730 | 964.6 s |
| PatternSearchCV, defaults (Exp. 10/this run) | 22 | 5.09 | 805.038 | 1157.6 s |
| **BayesHalvingSearchCV** (this run) | 28 | **3.89** | 805.730 | 993.2 s |

`BayesHalvingSearchCV` reached the *exact same MAE* Optuna's GPSampler found
at 100% data (805.730), at **3.89 full-fit equivalents - a 74% reduction from
Optuna's 15.00**, and 24% fewer equivalents than `PatternSearchCV`'s own best
(5.09), despite needing more raw evaluations (28 vs 22) to get there - the
bullseye multi-fidelity ladder is doing its job: 17 of the 28 evaluations ran
at the cheapest 0.5% rung, 8 more at the 10% rung after a single zone climb
(3 of those are the `promote_k=3` re-score of the top configs that triggered
the climb), and only the mandatory 3-row final polish touched full data.
Wall-clock
(993.2s vs 1157.6s, ratio 0.858) favors BHS too, but per this project's
established machine-noise rule (~15-25% is noise on this machine), that
difference is not a confidently-claimable win by itself - full-fit
equivalents is the primary, load-bearing metric, and there BHS's win is real
and substantial (well outside the noise floor).

`BayesHalvingSearchCV` did not find `PatternSearchCV`'s slightly better optimum
(805.038 at (4,130,17) vs BHS's 805.730 at (4,150,17)) - a 0.09% relative MAE
difference, i.e. both estimators are in the same optimum basin this project
keeps finding at low data fractions (Experiments 7-11), just at adjacent grid
points. Given GPProposer's own validation (`GP_Validation_vs_Optuna.ipynb`,
Experiment 12) showed it lands in this exact basin reliably, this reads as
expected between-estimator noise at n_starts=1, not a search-quality gap -
confirming this would need repeated seeds or the n_starts=4 follow-up arm.

**Mission success (spec section 0)**: "GP-quality answers (~805) at well
under 15 equivalents, at n_starts=1" - met, and BayesHalvingSearchCV also
beat PatternSearchCV's own equivalents count on this run, which was not a
required bar but is a strong additional result.